# Machine Learning Project Title
##### *Author: Slavena Peneva-Kargiou*  
##### *Course: SoftUni Machine Learning* 
##### *Instructor: Yordan Darakchiev*
##### *Date: June 2026*

In [15]:
import subprocess
subprocess.run(['pip', 'install', 'jieba'], check=True)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from scipy.stats import spearmanr, mannwhitneyu
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     GridSearchCV, KFold)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (mean_squared_error, mean_absolute_error,
                             r2_score, classification_report)
from sklearn.decomposition import PCA
from collections import Counter
import jieba
import re
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

# 1. Introduction

## 1.1 Motivation and Context
Children learning their first language acquire words in a consistent and well-documented order. Concrete nouns and social words come first - the things they can see, touch and point at. Pronouns, connectors and quantifiers are among the most common words in child-directed speech, yet children learn them last precisely because they refer to nothing in the physical world.

Language models learn differently. Their only signal is statistical: a word that appears frequently in many different contexts becomes well represented. There is no embodied grounding and no emotional attachment to the sound of a word before its meaning arrives.

In an earlier project, I examined this gap using English child vocabulary data from the Wordbank database and the BabyLM corpus - a curated collection of child-directed English text used to train language models. The finding was clear - corpus frequency predicted almost nothing about the order in which children acquire words, while concreteness was a strong predictor.

This project begins where thatF finding left off and was also inspired by personal experience. I learned Mandarin as an adult, deliberately, through structured study and largely through text. There was no pointing at things and no emotional connection to the language and its speakers. That experience feels structurally different from how children learn their first language and, in some ways, more similar to what a language model does. Both an adult L2 learner and a language model rely heavily on text and explicit pattern recognition. Neither acquires the language through the embodied emotionally-driven process that characterizes child L1 acquisition.

This project follows that intuition with data and ML models using Mandarin as the common language across all three learners and tries to answer the following question: Do the statistical features that predict word acquisition order differ between children learning Mandarin as L1 and adults learning it as L2, and does the adult L2 pattern more closely resemble a language model's frequency-driven learning signal?

## 1.3 The Three Learners
All three learners in this project are working on the same language - Mandarin. Apart from my personal experience learning it as an adult, it is also used for practical and conceptual reasons. Practically, all required datasets are available in Mandarin. Conceptually, it is typologically distant from the Indo-European languages, making it an interesting case.
### 1.3.1 Children
I am using Wordbank Mandarin (Beijing) CDI data. Acquisition order is measured as age of acquisition (AoA) - the youngest age in months at which at least 50% of children produce a given word.
### 1.3.2 Adult L2 learners
HSK level is the vocabulary grading from China's official Mandarin proficiency framework, the Hanyu Shuiping Kaoshi. I use the old HSK system (6 levels, approximately 5000 words), whereby HSK level 1 contains the 150 most basic words, while HSK level 6 is the most advanced. Important limitation stated upfront: the HSK level reflects curriculum design decisions based on expert judgement about what adult learners need to know, not on empirical measurement of what they actually learn first.
### 1.3.3 Language model
It is represented by word frequency in the Chinese BabyLM corpus, a 100-million-token Mandarin corpus designed for the Chinese BabyLM Challenge and built on the same developmental plausibility features as the English BabyLM corpus. Frequency is the model's primary learning signal.

## 1.4 Positioning Statement
The BabyLM challenge (Warstadt et al., 2023 onwards) and its Chinese counterpart (Chinese BabyLM Challenge, Hu et al.) train language models on developmentally plausible data, using child acquisition as the benchmark for human-like language learning. This project does not enter either challenge but engages with their central premise, and furthermore asks whether adult L2 acquisition might be a natural complement or alternative reference point to child acquisition when thinking about how models learn language.

No such three-way comparison - children, adult L2 learners and language models, on the same language, appears to exist in the published literature. This is an exploratory study. The datasets are real, the methods are careful and the conclusions are stated honestly. Every limitation is acknowledged explicitly.

## 1.5 Theoretical Framework and Hypotheses
The three learners in this project occupy different positions on a spectrum defined by how much they rely on embodied social experience versus statistical patterns in text when acquiring language.

Children learning L1 are at the fully embodied end. Their early vocabulary is shaped by what they can perceve and interact, which is why concreteness drives acquisition order.

Adult L2 learners occupy a middle position. They bring full conceptual knowledge of the world to the new language, but their learning is primarily text driven and explicit.

Language models are at the fully statistical end, frequency being their only signal.

This suggests three testable predictions:

$
H_1: \text{Frequency is a stronger predictor of adult L2 acquisition order (HSK level) than of child L1 acquisition order (AoA).}
$

$
H_2: \text{Concreteness is a weaker predictor of adult L2 acquisition order than of child L1 acquisition order.}
$

$
H_3: \text{The overall feature importance profile of the adult L2 model resembles the model's frequency-based learning signal more closely than the child model does.}
$

These are exploratory predictions, not formal null hypotheses. The project follows them with data and reports what it finds.

# 2. The Datasets
Four datasets are used. Two provide the target variables - AoA for children (Wordbank) and HSK level for adult L2 learners. Two provide features shared across both datasets - frequency (Chinese BabyLM corpus) and concreteness (Xu & Li, 2020). Word length in characters is derived directly from the word strings and requires no external source.

The datasets are merged into two parallel analysis datasets:
- Child dataset: Wordbank words with AoA and all available features
- Adult dataset: HSK words with HSK level and all available features


## 2.1 Wordbank Mandarin

In [5]:
df_wordbank = pd.read_csv('wordbank_mandarin_items.csv',
                           encoding='utf-8', quoting=3)
df_wordbank.columns = [c.strip('"') for c in df_wordbank.columns]
df_wordbank['item_definition'] = df_wordbank['item_definition'].str.strip('"')
df_wordbank['category'] = df_wordbank['category'].str.strip('"')
df_wordbank['word'] = df_wordbank['item_definition'].str.strip()

age_cols = [str(a) for a in range(16, 31)]

def compute_aoa(row, threshold=0.5):
    """Return the first age (months) at which >= 50% of children produce the word."""
    for age in age_cols:
        try:
            if float(row[age]) >= threshold:
                return int(age)
        except:
            pass
    return np.nan

df_wordbank['aoa'] = df_wordbank.apply(compute_aoa, axis=1)
df_aoa = df_wordbank[['word', 'category', 'aoa']].copy()

print(f"Total Wordbank words:              {len(df_aoa)}")
print(f"Words with AoA defined (≥50%):    {df_aoa['aoa'].notna().sum()}")
print(f"Words never reaching threshold:   {df_aoa['aoa'].isna().sum()}")
display(df_aoa.head(5))

Total Wordbank words:              799
Words with AoA defined (≥50%):    772
Words never reaching threshold:   27


,word,category,aoa
0,喂？,sounds,16.0
1,旺旺（狗叫）,sounds,16.0
2,喵（猫叫）,sounds,16.0
3,嘀嘀（汽车声）,sounds,17.0
4,哎哟,sounds,17.0


## 2.2 HSK Vocabulary Lists
The old HSK system (6 levels) is used. The words appearing in multiple levels are resolved by keeping the lowest level - since level 1 means learned first, this seems to be the most logical and methodologically appropriate choice.

In [6]:
hsk_dfs = []
for level in range(1, 7):
    df = pd.read_csv(f'hsk{level}.csv',
                     header=None, names=['word', 'pinyin', 'english'])
    df['hsk_level'] = level
    hsk_dfs.append(df)

df_hsk = pd.concat(hsk_dfs, ignore_index=True)

# Resolve duplicates — keep lowest level
df_hsk = df_hsk.sort_values('hsk_level').drop_duplicates(
    subset='word', keep='first').reset_index(drop=True)

print(f"Total HSK words after deduplication: {len(df_hsk)}")
print(f"\nWords per level:")
print(df_hsk['hsk_level'].value_counts().sort_index())
display(df_hsk.head(5))

Total HSK words after deduplication: 4993

Words per level:
hsk_level
1     150
2     147
3     298
4     598
5    1300
6    2500
Name: count, dtype: int64


,word,pinyin,english,hsk_level
0,开,kāi,to open,1
1,看,kàn,to see,1
2,看见,kàn jiàn,to see,1
3,块,kuài,lump (of earth),1
4,来,lái,to come,1


## 2.3 Chinese BabyLM Corpus - Computing Word Frequencies
The Chinese BabyLM corpus is a 100-million-token Mandarin dataset built from child-directed speech, children's books, educational text and subtitles, designed for the Chinese BabyLM Challenge. It is the Mandarin equivalent of the English BabyLM corpus and provides the model's learning signal throughout this project.

Word frequencies are computed using jieba, the standard Chinese word segmentation library, which splits unsegmented Chinese text into words. Because this computation takes approximately 40 minutes on the full corpus, the results are saved to a csv file. This precomputed file is included in the project repository so the notebook can be run without repeating the computation.

In [16]:
FREQ_CACHE = 'babylm_zh_frequencies.csv'

if os.path.exists(FREQ_CACHE):
    # Load pre-computed frequencies — no need to reprocess
    df_freq = pd.read_csv(FREQ_CACHE)
    print(f"Loaded pre-computed frequencies: {len(df_freq):,} unique words")
else:
    # Compute from scratch — only needed once
    print("Computing frequencies from Chinese BabyLM corpus...")
    print("This will take approximately 40 minutes.")

    df_babylm = pd.read_parquet('train-00000-of-00001.parquet')
    print(f"Corpus: {len(df_babylm):,} documents")
    print(f"Categories: {df_babylm['category'].value_counts().to_dict()}")

    word_counts = Counter()
    for i, text in enumerate(df_babylm['text']):
        words = jieba.lcut(str(text))
        word_counts.update(words)
        if (i + 1) % 10000 == 0:
            print(f"  Processed {i+1:,} / {len(df_babylm):,} documents...")

    df_freq = pd.DataFrame({
        'word':      list(word_counts.keys()),
        'frequency': list(word_counts.values())
    })

    # Save for future use
    df_freq.to_csv(FREQ_CACHE, index=False, encoding='utf-8-sig')
    print(f"Saved to {FREQ_CACHE}")

print(f"\nFrequency range: {df_freq['frequency'].min():,} – "
      f"{df_freq['frequency'].max():,}")
print(f"Total tokens: {df_freq['frequency'].sum():,}")
display(df_freq.sort_values('frequency', ascending=False).head(10))

Computing frequencies from Chinese BabyLM corpus...
This will take approximately 40 minutes.


Building prefix dict from the default dictionary ...


Corpus: 183,598 documents
Categories: {'educational': 74930, 'child-directed-speech': 42884, 'child-books': 25175, 'subtitles': 20293, 'child-available-speech': 20249, 'child-wiki': 67}


Dumping model to file cache /var/folders/_v/19_my0lx2lb3wvl7hr_wcvy80000gp/T/jieba.cache
Loading model cost 0.608 seconds.
Prefix dict has been built successfully.


  Processed 10,000 / 183,598 documents...
  Processed 20,000 / 183,598 documents...
  Processed 30,000 / 183,598 documents...
  Processed 40,000 / 183,598 documents...
  Processed 50,000 / 183,598 documents...
  Processed 60,000 / 183,598 documents...
  Processed 70,000 / 183,598 documents...
  Processed 80,000 / 183,598 documents...
  Processed 90,000 / 183,598 documents...
  Processed 100,000 / 183,598 documents...
  Processed 110,000 / 183,598 documents...
  Processed 120,000 / 183,598 documents...
  Processed 130,000 / 183,598 documents...
  Processed 140,000 / 183,598 documents...
  Processed 150,000 / 183,598 documents...
  Processed 160,000 / 183,598 documents...
  Processed 170,000 / 183,598 documents...
  Processed 180,000 / 183,598 documents...
Saved to babylm_zh_frequencies.csv

Frequency range: 1 – 9,696,699
Total tokens: 101,343,320


,word,frequency
12,,9696699
5,的,3898022
445047,",",2669196
239,我,2415696
26,了,2255739
242,你,2078294
445046,\n,1873593
159,是,1403442
445045,。,1385388
508992,$,1049841


## 2.4 Xu & Li Concreteness Norms
Xu & Li (2020) provide concreteness ratins for 9877 two-character Chinese words collected from native Mandarin speakers. 

One important preprocessing step: the Xu & Li scale runs from 1 (very concrete) to 5 (very abstract) - the inverse of standard direction used in most English-language research. To ensure consistency with my earlier project, scores are reversed before the analysis:
$$concreteness = 6 - concreteness__original$$
After reversal, a score of 5 indicates maximum concreteness and a score of 1 indicates maximum abstractness.

In [19]:
df_concrete = pd.read_excel(
    'Concretenss_Ratings_of_9877_Two_Character_Chinese_Words.xlsx')
df_concrete = df_concrete.rename(columns={
    'Word':                  'word',
    'Mean of Valid Ratings': 'concreteness_raw'
})
df_concrete['word'] = df_concrete['word'].str.strip()

# Reverse scale: original 1=concrete, 5=abstract → reversed 5=concrete, 1=abstract
df_concrete['concreteness'] = 6 - df_concrete['concreteness_raw']

print(f"Concreteness norms: {len(df_concrete):,} words")
print(f"Original scale:  {df_concrete['concreteness_raw'].min():.2f} – "
      f"{df_concrete['concreteness_raw'].max():.2f}  (1=concrete, 5=abstract)")
print(f"Reversed scale:  {df_concrete['concreteness'].min():.2f} – "
      f"{df_concrete['concreteness'].max():.2f}  (1=abstract, 5=concrete)")

# Verify reversal is sensible
print("\nMost concrete words after reversal (highest scores):")
display(df_concrete.nlargest(5, 'concreteness')[['word','concreteness_raw','concreteness']])
print("\nMost abstract words after reversal (lowest scores):")
display(df_concrete.nsmallest(5, 'concreteness')[['word','concreteness_raw','concreteness']])

Concreteness norms: 9,877 words
Original scale:  1.04 – 4.56  (1=concrete, 5=abstract)
Reversed scale:  1.44 – 4.96  (1=abstract, 5=concrete)

Most concrete words after reversal (highest scores):


,word,concreteness_raw,concreteness
9422,鸡蛋,1.038462,4.961538
9468,兔子,1.038462,4.961538
5997,山羊,1.040000,4.960000
4081,菠菜,1.041667,4.958333
5127,企鹅,1.064516,4.935484



Most abstract words after reversal (lowest scores):


,word,concreteness_raw,concreteness
8044,仿佛,4.560000,1.440000
3022,命运,4.517241,1.482759
8320,理想,4.434783,1.565217
53,笼统,4.423077,1.576923
507,抽象,4.423077,1.576923


## 2.5 Building the Two Parallel Datasets

## 2.6 What Gets Lost in the Merge and Why

# 4. Exploratory Data Analysis

## 4.1 AoA and HSK Level Distributions

## 4.2 Frequency Distributions and Zipf's Law

## 4.3 Concreteness Distributions

## 4.4 Spearman Correlations

## 4.5 Scatter Plots

# 5. Machine Learning Models

## 5.1 Feature Set and Setup

## 5.2 Model 1 - Predicting Child AoA 

## 5.3 Model 2 - Predicting HSK Level

## 5.4 Lasso - Feature Selection ???

# 6. The Three Learners - Feature Importance Comparison

## 6.1 Coefficient Comparison

## 6.2 Feature Importances

## 6.3 PCA

## 6.4 Summary: Where Does the Adult L2 Learner Sit?

# 7. Discussion and Limitations
## 7.1 What the Models Found
## 7.2 Limitations
## 7.3 What a Larger Study Might Look Like

# 8. Conclusion

# 9. References